# Coastal Ecuador Biophysical Productivity — Results Figures

Generates the result figures from the manuscript's Results section: long-term
trends, interannual anomalies, the climatological seasonal cycle, spatial
climatology, and pixel-by-pixel correlation maps between chlorophyll-a and
each physical variable (wind, SST, MSL, SSS).

**Structure**
- 0. Setup: imports, Google Drive mount, data loading (time series + spatial cubes)
- 1. Long-term temporal evolution (5 panels)
- 2. Interannual anomalies (5 panels)
- 3. Climatological seasonal cycle (5 panels)
- 4. Climatological spatial distribution (5x3 grid, unified color scale)
- 5. Pixel-by-pixel spatial correlations (Pearson and Spearman; full period,
  dry season, wet season)

Plot titles are kept generic (no embedded figure numbers or captions); figure
numbering and captions are assigned in the manuscript text.


## 0. Initial setup

All notebook imports in one place, and Google Drive mounting.

In [ ]:
# Libraries needed for the whole notebook (time series, maps, and statistics)
from google.colab import drive
import os
import glob
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from matplotlib import ticker as _mticker
from matplotlib.patches import FancyBboxPatch
from scipy import stats

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
except ImportError:
    print("Installing Cartopy...")
    !pip install Cartopy
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

warnings.filterwarnings('ignore')

# Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Root folder containing the raw .nc files produced by 01_data_download.ipynb.
# Edit this path to match your own Drive layout if needed.
DATA_ROOT = "/content/drive/MyDrive/oceanographic_data/raw"


Check that the data folders exist in Drive (diagnostic only, generates no figures).

In [ ]:
base_path = DATA_ROOT

if os.path.exists(base_path):
    print("Main folder found.")
    print("Contents:")
    print(os.listdir(base_path))

    # Check CMEMS subfolder
    cmems_path = os.path.join(base_path, "CMEMS_weekly")
    if os.path.exists(cmems_path):
        print(f"\nCMEMS subfolder found. Contents:")
        print(os.listdir(cmems_path))
    else:
        print(f"'CMEMS_weekly' not found. Check whether DATA_ROOT is set correctly.")
else:
    print("Main folder not found. Check that DATA_ROOT points to the correct Drive folder.")


### 0.1 Loading time series (spatial average per week)

Defines `load_data()`, the paths (`folders`, `var_names`), and loads the `data_store` DataFrame with the 5 variables. Feeds **Figures 3, 4, and 5**.

In [ ]:
# Data folder paths in Drive (must match the layout produced by 01_data_download.ipynb)
base_dir = DATA_ROOT
cmems_dir = os.path.join(base_dir, "CMEMS_weekly")

folders = {
    "SST":  os.path.join(cmems_dir, "SST"),
    "SSS":  os.path.join(cmems_dir, "SSS"),
    "MSL":  os.path.join(cmems_dir, "MSL"),
    "WIND": os.path.join(base_dir, "WIND"),
    "CHL":  os.path.join(base_dir, "CHL"),
}

var_names = {
    "SST":    "thetao",
    "SSS":    "so",
    "MSL":    "zos",
    "WIND": "wind_speed_10m",
    "CHL":   "chlor_a",
}

def load_data(var_key):
    """Loads and spatially averages (per week) a variable from its .nc files in Drive."""
    folder = folders[var_key]
    var_name = var_names[var_key]

    if not os.path.exists(folder):
        print(f"Error: folder not found: {folder}")
        return None

    files = sorted(glob.glob(os.path.join(folder, "**", "*.nc"), recursive=True))
    if not files:
        files = sorted(glob.glob(os.path.join(folder, "**", "*.NC"), recursive=True))

    if not files:
        print(f"Warning: folder '{var_key}' exists but contains no .nc files.")
        print(f"  Folder contents (first 5): {os.listdir(folder)[:5]} ...")
        return None

    print(f"{var_key}: found {len(files)} files, processing...")

    start_date_wind = pd.Timestamp("2002-07-04")
    records = []

    for f in files:
        try:
            with xr.open_dataset(f) as ds:
                if var_key == "WIND":
                    try:
                        week_str = os.path.basename(f).split("_week_")[-1].split(".")[0]
                        week_idx = int(week_str)
                        t = start_date_wind + pd.to_timedelta(7 * (week_idx - 1), unit="D")
                    except:
                        continue
                else:
                    if "time" in ds.coords:
                        t = pd.to_datetime(ds["time"].values[0])
                    else:
                        continue

                if var_name in ds:
                    data_array = ds[var_name]
                    dims_to_slice = {}
                    for d in data_array.dims:
                        if d not in ['lat', 'lon', 'latitude', 'longitude', 'time']:
                            dims_to_slice[d] = 0
                    if dims_to_slice:
                        data_array = data_array.isel(dims_to_slice)

                    mean_val = data_array.mean().item()
                    records.append({"time": t, "value": mean_val})
        except Exception:
            continue

    if not records:
        print(f"{var_key}: files were found but could not be read (check format or naming).")
        return None

    df = pd.DataFrame(records).sort_values("time").reset_index(drop=True)
    return df


In [ ]:
# Visual configuration and order of the 5 variables for Figures 3, 4, and 5
variables_ordenadas = ["WIND", "SST", "MSL", "SSS", "CHL"]

var_props = {
    "WIND": {"color": "#1f77b4", "label": "Wind (m/s)",            "title": "a) Forcing: 10-m Wind"},
    "SST":    {"color": "#d62728", "label": "Surface temp. (°C)",    "title": "b) Physical: Temperature (SST)"},
    "MSL":    {"color": "#17becf", "label": "Mean sea level (m)",    "title": "c) Physical: Sea Level (MSL)"},
    "SSS":    {"color": "#9467bd", "label": "Surface salinity (PSU)","title": "d) Physical: Salinity (SSS)"},
    "CHL":   {"color": "#2ca02c", "label": "Chlorophyll-a (mg/m³)", "title": "e) Biological: Chlorophyll-a"}
}

data_store = {}
print("Preparing data for the 5 panels...")

for var in variables_ordenadas:
    df = load_data(var)
    if df is not None and not df.empty:
        df = df.copy()
        df["time"] = pd.to_datetime(df["time"])
        df["year"] = df["time"].dt.year
        df["month"] = df["time"].dt.month
        data_store[var] = df
    else:
        print(f"Skipping {var}: no data loaded.")


### 0.2 Loading spatial cubes (lat-lon-time)

Defines `load_spatial_cube_robust()` and loads `spatial_data`, needed for **Figures 6 through 12**.

In [ ]:
# --- MAP VISUAL CONFIGURATION ---
map_configs = {
    "WIND": {"cmap": "viridis",       "label": "m/s",   "title": "Mean Wind"},
    "SST":    {"cmap": "RdYlBu_r",      "label": "°C",    "title": "Mean SST"},
    "MSL":    {"cmap": "Blues",         "label": "m",     "title": "Mean MSL"},
    "SSS":    {"cmap": "viridis",       "label": "PSU",   "title": "Mean SSS"},
    "CHL":   {"cmap": "nipy_spectral", "label": "mg/m³", "title": "Mean Chlorophyll-a"}
}

def load_spatial_cube_robust(var_key, freq='M'):
    """Loads the space-time cube (lat, lon, time) of a variable, resampled to freq."""
    folder = folders[var_key]
    var_name = var_names[var_key]
    files = sorted(glob.glob(os.path.join(folder, "**", "*.nc"), recursive=True))
    if not files:
        files = sorted(glob.glob(os.path.join(folder, "**", "*.NC"), recursive=True))
    if not files:
        return None

    print(f"Loading: {var_key}...")
    try:
        if var_key == "WIND":
            datasets = []
            start_date = pd.Timestamp("2002-07-04")
            # Limited loading for speed. Remove [:300] if you have plenty of RAM.
            files_to_load = files[:300]
            for f in files_to_load:
                try:
                    ds = xr.open_dataset(f)
                    week_str = os.path.basename(f).split("_week_")[-1].split(".")[0]
                    t = start_date + pd.to_timedelta(7 * (int(week_str) - 1), unit="D")
                    ds = ds.expand_dims(time=[t])
                    if 'latitude' in ds.dims:
                        ds = ds.rename({'latitude': 'lat', 'longitude': 'lon'})
                    datasets.append(ds)
                except:
                    pass
            if not datasets:
                return None
            ds_full = xr.concat(datasets, dim='time')
        else:
            ds_full = xr.open_mfdataset(files, combine='by_coords', parallel=True)
            if 'latitude' in ds_full.dims:
                ds_full = ds_full.rename({'latitude': 'lat', 'longitude': 'lon'})

        da = ds_full[var_name]
        dims_to_drop = [d for d in da.dims if d not in ['time', 'lat', 'lon']]
        if dims_to_drop:
            da = da.isel({d: 0 for d in dims_to_drop})
        da = da.sortby('time').resample(time=freq).mean()
        return da
    except Exception:
        return None

spatial_data = {}
vars_to_load = ["CHL", "WIND", "SST", "MSL", "SSS"]
print("--- Starting spatial cube loading ---")
for var in vars_to_load:
    da = load_spatial_cube_robust(var)
    if da is not None:
        spatial_data[var] = da


### 0.3 Spatial helper functions

Seasonal subset (`subset_season`), map axis setup (`setup_map_ax`), and target grids (`grid_high` for distribution, `grid_med` for correlations).

In [ ]:
# Shared configuration for the spatial figures (Figures 6 through 12)
plot_order_maps = ["WIND", "SST", "MSL", "SSS", "CHL"]
physical_vars = ["WIND", "SST", "MSL", "SSS"]

dry_months = [6, 7, 8, 9, 10, 11]   # June-November (dry season)
wet_months = [12, 1, 2, 3, 4, 5]    # December-May (wet season)

# Target grids: high resolution for distribution, intermediate for correlations
grid_high = spatial_data["CHL"].isel(time=0)
grid_med  = spatial_data["SST"].isel(time=0)

def subset_season(da, season=None):
    """season: None (full period), 'dry' (dry season), or 'wet' (wet season)."""
    if season is None:
        return da
    months = dry_months if season == "dry" else wet_months
    return da.where(da["time"].dt.month.isin(months), drop=True)

def setup_map_ax(ax, show_left_labels=True):
    """Coastline, land, and coordinate grid for a map axis (Cartopy)."""
    ax.add_feature(cfeature.COASTLINE, linewidth=1.2, zorder=5)
    ax.add_feature(cfeature.LAND, facecolor="#e0e0e0", zorder=4)
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="gray",
                       alpha=0.5, linestyle=":", zorder=6)
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 8}
    gl.ylabel_style = {"size": 8}
    if not show_left_labels:
        gl.left_labels = False
    return gl


### 0.4 Pixel-by-pixel correlation and global p-value functions

`pearson_r_p_map` / `spearman_r_p_map_rank`: per-pixel correlation and p-value. `sig_fraction`: % of significant pixels (p<0.05). `area_mean_anomaly_corr`: *global* correlation and p-value (spatial average, reported in each panel subtitle as `global r=...`).

In [ ]:
def pearson_r_p_map(da1, da2):
    """
    r: xr.corr pixel by pixel.
    p: t-test approximation using effective n per pixel:
       t = r*sqrt((n-2)/(1-r^2)), p = 2*sf(|t|, df=n-2)
    """
    common_time = np.intersect1d(da1.time.values, da2.time.values)
    da1s = da1.sel(time=common_time)
    da2s = da2.sel(time=common_time)

    valid = np.isfinite(da1s) & np.isfinite(da2s)
    n = valid.sum("time")

    r = xr.corr(da1s.where(valid), da2s.where(valid), dim="time")

    df = (n - 2).astype(float)
    denom = xr.where(np.abs(r) < 1, (1 - r**2), np.nan)
    t = r * np.sqrt(df / denom)

    p = xr.apply_ufunc(
        lambda tt, dff: 2 * stats.t.sf(np.abs(tt), dff),
        t, df,
        input_core_dims=[[], []],
        output_core_dims=[[]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
    )

    return r, p, n


def spearman_r_p_map_rank(da1, da2):
    """
    Pixel-by-pixel Spearman via ranks over time:
    rho = pearson( rank(x), rank(y) )
    p: t approximation using n per pixel.
    """
    common_time = np.intersect1d(da1.time.values, da2.time.values)
    da1s = da1.sel(time=common_time).chunk({"time": -1})
    da2s = da2.sel(time=common_time).chunk({"time": -1})

    valid = np.isfinite(da1s) & np.isfinite(da2s)
    n = valid.sum("time")

    r1 = da1s.where(valid).rank("time")
    r2 = da2s.where(valid).rank("time")

    rho = xr.corr(r1, r2, dim="time")

    df = (n - 2).astype(float)
    denom = xr.where(np.abs(rho) < 1, (1 - rho**2), np.nan)
    t = rho * np.sqrt(df / denom)

    p = xr.apply_ufunc(
        lambda tt, dff: 2 * stats.t.sf(np.abs(tt), dff),
        t, df,
        input_core_dims=[[], []],
        output_core_dims=[[]],
        vectorize=True,
        dask="parallelized",
        dask_gufunc_kwargs={"allow_rechunk": True},
        output_dtypes=[float],
    )

    return rho, p, n


def sig_fraction(p_map, alpha=0.05):
    """Percentage of pixels with p < alpha."""
    finite = np.isfinite(p_map.values)
    if finite.sum() == 0:
        return np.nan
    return 100 * np.nansum((p_map.values < alpha) & finite) / np.nansum(finite)


In [ ]:
def area_mean_ts(da):
    """Spatial average over (lat, lon), ignoring NaNs."""
    return da.mean(dim=("lat", "lon"), skipna=True)

def area_mean_anomaly_corr(da_chl, da_phys, method="pearson"):
    """
    GLOBAL correlation and p-value per panel:
    - Interpolate to grid_med (for spatial coherence)
    - Time series = spatial average
    - Anomalies = series - mean(series) (over the selected period)
    - Correlation (Pearson or Spearman)
    """
    common_time = np.intersect1d(da_chl.time.values, da_phys.time.values)
    if len(common_time) < 10:
        return np.nan, np.nan, len(common_time)

    chl = da_chl.sel(time=common_time).interp_like(grid_med).chunk({"time": -1})
    phy = da_phys.sel(time=common_time).interp_like(grid_med).chunk({"time": -1})

    chl_ts = area_mean_ts(chl).compute()
    phy_ts = area_mean_ts(phy).compute()

    x = chl_ts.values
    y = phy_ts.values
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 10:
        return np.nan, np.nan, len(x)

    x_anom = x - np.nanmean(x)
    y_anom = y - np.nanmean(y)

    if method == "pearson":
        r, p = stats.pearsonr(x_anom, y_anom)
    else:
        r, p = stats.spearmanr(x_anom, y_anom)

    return r, p, len(x_anom)


## 1. Long-term temporal evolution (2002-2025), 5 panels

a) 10-m wind, b) SST, c) MSL, d) SSS, e) chlorophyll-a. Gray = weekly data,
colored line = quarterly moving average, orange dots = annual average,
dashed line = linear trend.

In [ ]:
# Temporal evolution (5 panels)
if data_store:
    fig, axes = plt.subplots(5, 1, figsize=(12, 14), sharex=True)

    for i, var in enumerate(variables_ordenadas):
        if var not in data_store:
            continue

        df = data_store[var].sort_values("time").copy()
        ax = axes[i]
        props = var_props[var]

        # Annual data (annual average) for the trend
        annual = df.groupby("year", as_index=True)["value"].mean().dropna()
        annual_years = annual.index.values.astype(float)
        annual_dates = [pd.Timestamp(f"{y}-07-01") for y in annual.index]

        # Annual linear trend: value ~ year
        slope, intercept, r, p, se = stats.linregress(annual_years, annual.values)
        trend_line = intercept + slope * annual_years

        # 1) Raw data (background)
        ax.plot(df["time"], df["value"], color="gray", alpha=0.3, linewidth=0.5)

        # 2) Smoothing (moving average) ~ 12 weeks (≈ quarterly if weekly)
        df["smooth"] = df["value"].rolling(window=12, center=True).mean()
        ax.plot(
            df["time"], df["smooth"],
            color=props["color"], linewidth=1.5,
            label="Trend (Quarterly)"
        )

        # 3) Annual points (annual average) + trend line
        ax.plot(annual_dates, annual.values, "o", color="orange", markersize=4,
                zorder=5, label="Annual average")
        ax.plot(
            annual_dates, trend_line,
            "--", color="black", linewidth=1.2,
            label=f"Trend ({slope:.4f}/yr)"
        )

        # Specific adjustment for chlorophyll (clip peaks)
        if var == "CHL":
            ymax = df["value"].quantile(0.99)  # ignore top 1%
            if np.isfinite(ymax) and ymax > 0:
                ax.set_ylim(0, ymax)

        # Labels / internal title
        ax.set_ylabel(props["label"], fontweight="bold", fontsize=9)
        ax.text(
            0.01, 0.9, props["title"],
            transform=ax.transAxes,
            fontweight="bold", fontsize=10,
            bbox=dict(facecolor="white", alpha=0.7, edgecolor="none")
        )

        ax.grid(True, alpha=0.4, linestyle="--")

        # Legend in every panel, upper right
        ax.legend(
            loc="upper right",
            frameon=True,
            fontsize=7,
            handlelength=2,
            borderpad=0.3,
            labelspacing=0.3
        )

    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    plt.show()

else:
    print("No data available to plot.")


## 2. Interannual anomalies (2002-2025), 5 panels

Anomalies relative to the full-period mean, with ENSO periods highlighted.

In [ ]:
# Interannual anomalies (5 panels) with ENSO period highlighting
if data_store:
    fig, axes = plt.subplots(5, 1, figsize=(12, 12), sharex=True)

    # ENSO periods to highlight (adjust the range if you want a different one)
    enos_periods = [
        {"start": 2015, "end": 2016, "label": "El Niño 2015–2016"},
        {"start": 2023, "end": 2024, "label": "El Niño 2023–2024"}
    ]

    for i, var in enumerate(variables_ordenadas):
        if var not in data_store:
            continue

        df = data_store[var].copy()
        ax = axes[i]
        props = var_props[var]

        # Interannual anomalies (annual average - overall mean)
        annual = df.groupby("year")["value"].mean()
        anomalies = annual - annual.mean()

        # Colors (Red=Positive, Blue=Negative)
        colors = ["#d62728" if v > 0 else "#1f77b4" for v in anomalies.values]

        # Bars
        ax.bar(anomalies.index, anomalies.values, color=colors, alpha=0.8, zorder=2)
        ax.axhline(0, color="black", linewidth=0.8, zorder=3)

        # Highlight ENSO periods with shaded bands and boundary lines
        for p in enos_periods:
            # shaded band
            ax.axvspan(p["start"] - 0.5, p["end"] + 0.5, color="gray", alpha=0.12, zorder=1)

            # boundary lines (so it looks "marked with lines")
            ax.axvline(p["start"], color="gray", linestyle=":", linewidth=1, alpha=0.6, zorder=4)
            ax.axvline(p["end"],   color="gray", linestyle=":", linewidth=1, alpha=0.6, zorder=4)

            # label only on the first panel (to avoid repeating it 5 times)
            if i == 0:
                ax.text(
                    x=(p["start"] + p["end"]) / 2,
                    y=ax.get_ylim()[1] * 0.92,
                    s=p["label"],
                    ha="center", va="top",
                    fontsize=9, fontweight="bold",
                    color="gray"
                )

        # Labels and style
        ax.set_ylabel("Anomaly", fontsize=9)
        ax.text(
            0.01, 0.85, props["title"],
            transform=ax.transAxes,
            fontweight="bold", fontsize=10,
            bbox=dict(facecolor="white", alpha=0.7, edgecolor="none")
        )
        ax.grid(axis="x", linestyle=":", alpha=0.5, zorder=0)

    axes[-1].set_xlabel("Year")
    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    plt.show()


## 3. Climatological seasonal cycle (monthly averages, 2002-2025), 5 panels

Vertical dashed lines mark the boundaries between the wet season (Dec-May)
and the dry season (Jun-Nov).

In [ ]:
# Seasonal cycle (5 panels)
if data_store:
    fig = plt.figure(figsize=(14, 15))

    gs = fig.add_gridspec(5, 1, hspace=0.12, top=0.97, bottom=0.09)
    axes = [fig.add_subplot(gs[i]) for i in range(5)]

    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    season_boundaries = [4.5, 10.5]

    for i, var in enumerate(variables_ordenadas):
        if var not in data_store: continue

        df = data_store[var]
        ax = axes[i]
        props = var_props[var]

        monthly_clim = df.groupby('month')['value'].mean()

        ax.plot(months, monthly_clim.values, '-o', color=props["color"],
                linewidth=2.5, markersize=7)

        ymin = min(monthly_clim.values)
        margin = (max(monthly_clim.values) - ymin) * 0.1
        base_line = ymin - margin if ymin > 0 else ymin + margin
        ax.fill_between(months, monthly_clim.values, base_line,
                        color=props["color"], alpha=0.15)

        for xpos in season_boundaries:
            ax.axvline(x=xpos, color="#555555", linewidth=1.4,
                       linestyle=(0, (6, 4)), zorder=3)

        ax.set_ylabel(props["label"], fontsize=13)
        ax.tick_params(axis='y', labelsize=12)
        ax.grid(True, alpha=0.3)
        ax.text(0.02, 0.85, props["title"], transform=ax.transAxes,
                fontweight='bold', fontsize=10)

        if i < 4:
            ax.tick_params(axis='x', labelbottom=False)
        else:
            ax.tick_params(axis='x', labelsize=13)

    # ------------------------------------------------------------------
    # Season bar below the x-axis
    # ------------------------------------------------------------------
    ax_bar = fig.add_axes([0.125, 0.015, 0.775, 0.055])
    ax_bar.set_xlim(-0.5, 11.5)
    ax_bar.set_ylim(0, 1)
    ax_bar.axis('off')

    # Wet season block 1: Jan–May
    ax_bar.add_patch(FancyBboxPatch((-0.5, 0.35), 5.0, 0.45,
                                    boxstyle="round,pad=0.05",
                                    fc="white", ec="#555555", lw=1.2))
    ax_bar.text(2.0, 0.57, "Wet season (Dec–May)",
                ha='center', va='center', fontsize=11,
                color="#333333", fontweight='500')

    # Dry season: Jun–Nov
    ax_bar.add_patch(FancyBboxPatch((4.5, 0.35), 6.0, 0.45,
                                    boxstyle="round,pad=0.05",
                                    fc="white", ec="#555555", lw=1.2))
    ax_bar.text(7.5, 0.57, "Dry season (Jun–Nov)",
                ha='center', va='center', fontsize=11,
                color="#333333", fontweight='500')

    # Wet season block 2: Dec
    ax_bar.add_patch(FancyBboxPatch((10.5, 0.35), 1.0, 0.45,
                                    boxstyle="round,pad=0.05",
                                    fc="white", ec="#555555", lw=1.2))
    ax_bar.text(11.0, 0.57, "Dec",
                ha='center', va='center', fontsize=9,
                color="#333333", fontweight='500')

    # Bottom brackets
    for x0, x1 in [(-0.45, 4.45), (4.55, 10.45), (10.55, 11.45)]:
        ax_bar.plot([x0, x1], [0.28, 0.28], color="#555555", lw=1.5,
                    solid_capstyle='round')
        ax_bar.plot([x0, x0], [0.28, 0.38], color="#555555", lw=1.5,
                    solid_capstyle='round')
        ax_bar.plot([x1, x1], [0.28, 0.38], color="#555555", lw=1.5,
                    solid_capstyle='round')

    plt.savefig("seasonal_cycle.png", dpi=150, bbox_inches="tight")
    plt.show()


## 4. Climatological spatial distribution (5x3 grid)

Rows = variables (wind, SST, MSL, salinity, chlorophyll-a). Columns = periods
(annual average, dry season, wet season). Unified color scale per row
(variable).

In [ ]:
# Rows    = variables (Wind, SST, MSL, Salinity, Chlorophyll-a)
# Columns = periods    (Annual average | Dry season jun-nov | Wet season dic-may)
# A single color range per variable, the same across the 3 columns (the bar
# covers the union of the 3 climatologies, so they remain comparable).

# Variables with outliers -> robust scale (2-98 percentiles). The rest: real min/max.
_robust_scale_vars = ["CHL", "SSS"]

# MANUALLY FIXED color ranges (override the automatic min/max calculation).
# These leave some "headroom" in the bar so the three columns —especially the
# wet season— are described better without saturating. Comment out a line to
# go back to that variable's automatic range, or edit the numbers as desired.
manual_scales = {
    "SST": (22, 27),       # °C
    "MSL": (0.21, 0.26),   # m
}

# Contour-line spacing per variable (isoline interval, in the variable's own
# units). Set a variable to None to skip contouring it.
contour_steps = {
    "WIND": 0.2,   # m/s
    "SST":    0.2,   # °C
    "MSL":    0.005, # m
    "SSS":    0.1,   # PSU
    "CHL":   0.10,  # mg/m³
}

# Contour-line color per variable, picked as a dark shade pulled from each
# variable's own colormap so the isolines blend with the map instead of
# competing with it (no labels, just thin reference lines).
contour_colors = {
    "WIND": plt.get_cmap("viridis")(0.05),
    "SST":    plt.get_cmap("RdYlBu_r")(0.95),
    "MSL":    plt.get_cmap("Blues")(0.95),
    "SSS":    plt.get_cmap("viridis")(0.05),
    "CHL":   plt.get_cmap("nipy_spectral")(0.85),
}

_row_labels = {
    "WIND": "a) 10-m Wind",
    "SST":    "b) Surface temperature (SST)",
    "MSL":    "c) Mean Sea Level (MSL)",
    "SSS":    "d) Surface salinity (SSS)",
    "CHL":   "e) Chlorophyll-a",
}
_col_titles  = ["Annual average", "Dry season (Jun–Nov)", "Wet season (Dec–May)"]
_col_seasons = [None, "dry", "wet"]

def _setup_map_ax_comb(ax, show_left_labels=True, show_bottom_labels=True):
    """Coastline + land + grid. Label lat only on the left column and lon only on the bottom row."""
    ax.add_feature(cfeature.COASTLINE, linewidth=1.0, zorder=5)
    ax.add_feature(cfeature.LAND, facecolor="#e0e0e0", zorder=4)
    gl = ax.gridlines(draw_labels=True, linewidth=0.4, color="gray",
                      alpha=0.5, linestyle=":", zorder=6)
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = show_left_labels
    gl.bottom_labels = show_bottom_labels
    gl.xlabel_style = {"size": 7}
    gl.ylabel_style = {"size": 7}
    return gl

def _clim_map(var, season):
    """Time average for the season, interpolated to the high-resolution grid (CHL)."""
    da = subset_season(spatial_data[var], season=season)
    return da.mean("time").interp_like(grid_high, method="linear")

def build_unified_scales():
    """
    scales: {var: (vmin, vmax)} -> a single range per variable, computed over the
            union of the 3 climatologies (annual + dry + wet).
    clims:  {var: {None/'dry'/'wet': DataArray}} -> reused to avoid recomputation.
    """
    scales, clims = {}, {}
    for var in plot_order_maps:
        if var not in spatial_data:
            continue
        maps = {s: _clim_map(var, s) for s in _col_seasons}
        clims[var] = maps
        allvals = np.concatenate([m.values.ravel() for m in maps.values()])
        allvals = allvals[np.isfinite(allvals)]
        if var in _robust_scale_vars:
            vmin = float(np.nanpercentile(allvals, 2))
            vmax = float(np.nanpercentile(allvals, 98))
        else:
            vmin = float(np.nanmin(allvals))
            vmax = float(np.nanmax(allvals))
        if var in manual_scales:
            vmin, vmax = manual_scales[var]
        scales[var] = (vmin, vmax)
    return scales, clims

def _contour_levels(vmin, vmax, step):
    """Build isoline levels spaced by `step`, snapped to round multiples of
    step so the lines fall on 'clean' values (e.g. 0.2, 0.4, 0.6... not
    0.17, 0.37...)."""
    start = np.ceil(vmin / step) * step
    stop = np.floor(vmax / step) * step
    if stop < start:
        return []
    n = int(round((stop - start) / step)) + 1
    return list(np.linspace(start, stop, n))

def plot_distribution_combined(savepath=None, dpi=300):
    """Climatological spatial distribution: grid of 5 (variables) x 3 (periods).
    Each map is overlaid with thin, unlabeled isolines (contour_steps) tinted
    to match each variable's own colormap, to make spatial gradients easier
    to read without competing visually with the color fill. Does not print a
    title inside the image; the caption is added in the manuscript."""
    scales, clims = build_unified_scales()
    vars_present = [v for v in plot_order_maps if v in spatial_data]
    nrow = len(vars_present)

    fig = plt.figure(figsize=(11, 2.6 * nrow))
    gs = fig.add_gridspec(nrow, 3,
                          left=0.17, right=0.98, top=0.97, bottom=0.05,
                          wspace=0.08, hspace=0.05)

    row_im, row_first_ax = {}, {}

    for r, var in enumerate(vars_present):
        conf = map_configs[var]
        vmin, vmax = scales[var]
        step = contour_steps.get(var)
        c_color = contour_colors.get(var, "black")
        for c, season in enumerate(_col_seasons):
            ax = fig.add_subplot(gs[r, c], projection=ccrs.PlateCarree())
            _setup_map_ax_comb(ax,
                               show_left_labels=(c == 0),
                               show_bottom_labels=(r == nrow - 1))
            da_map = clims[var][season]
            im = da_map.plot(
                ax=ax, transform=ccrs.PlateCarree(),
                cmap=conf["cmap"], vmin=vmin, vmax=vmax,
                add_colorbar=False
            )

            # --- Isolines (contours), no inline labels ---
            if step:
                levels = _contour_levels(vmin, vmax, step)
                if len(levels) >= 2:
                    da_map.plot.contour(
                        ax=ax, transform=ccrs.PlateCarree(),
                        levels=levels, colors=[c_color],
                        linewidths=0.5, alpha=0.7,
                        add_colorbar=False
                    )

            ax.set_title(_col_titles[c] if r == 0 else "", fontsize=11)
            if c == 0:
                row_im[var] = im
                row_first_ax[var] = ax

    fig.canvas.draw()  # cartopy fixes the aspect ratio when drawing; needed before reading positions

    # One color bar per row (left side) + rotated variable name
    for var in vars_present:
        pos = row_first_ax[var].get_position()
        cax = fig.add_axes([pos.x0 - 0.075, pos.y0 + pos.height * 0.10,
                            0.015, pos.height * 0.80])
        cb = fig.colorbar(row_im[var], cax=cax)
        cax.yaxis.set_ticks_position("left")
        cax.yaxis.set_label_position("left")
        cb.locator = _mticker.MaxNLocator(5)
        cb.update_ticks()
        cb.ax.tick_params(labelsize=7)
        cb.set_label(map_configs[var]["label"], fontsize=8)
        fig.text(pos.x0 - 0.145, pos.y0 + pos.height / 2.0, _row_labels[var],
                 rotation=90, va="center", ha="center",
                 fontsize=10, fontweight="bold")

    if savepath:
        fig.savefig(savepath, dpi=dpi, bbox_inches="tight")
        print(f"Figure saved to: {savepath}")
    plt.show()

    # Summary of the unified ranges used (useful for the caption / reviewers)
    print("Unified color ranges per variable (vmin, vmax):")
    for v in vars_present:
        print(f"  {v:7s}: {scales[v][0]:.3f}  ->  {scales[v][1]:.3f}  [{map_configs[v]['label']}]")
    return scales

In [ ]:
plot_distribution_combined(savepath="spatial_distribution.png", dpi=300)

## 5. Pixel-by-pixel spatial correlations

4 panels per figure (chlorophyll-a vs. wind, SST, MSL, SSS), Pearson and
Spearman, for the full period, the dry season, and the wet season.

In [ ]:
# Panel titles for the correlation figures (Chl-a vs. each physical variable)
panel_titles_corr = {
    "WIND": "a) Chl-a vs. 10-m Wind",
    "SST":    "b) Chl-a vs. SST",
    "MSL":    "c) Chl-a vs. MSL",
    "SSS":    "d) Chl-a vs. SSS",
}


In [ ]:
def plot_correlation_2x2(method="pearson", season=None):
    """
    Figures 7-12: pixel-by-pixel spatial correlation between Chl-a and each physical variable.
    method: 'pearson' or 'spearman'
    season: None (full period), 'dry' (dry season), or 'wet' (wet season)
    Does not print a title inside the image; the caption is added in the manuscript.
    """
    fig = plt.figure(figsize=(15, 10))

    # 3 columns: (left panel, right panel, colorbar)
    gs = GridSpec(
        2, 3, figure=fig,
        width_ratios=[1, 1, 0.06],
        wspace=0.04,
        hspace=0.18
    )

    da_chl = subset_season(spatial_data["CHL"], season=season)

    ims = []
    axes = []

    if method == "pearson":
        cbar_label = "Pearson Correlation (R)"
    else:
        cbar_label = "Spearman Rank (ρ)"
    cmap = "RdBu_r"
    vmin, vmax = -0.7, 0.7

    for i, p_var in enumerate(physical_vars):
        ax = fig.add_subplot(gs[i // 2, i % 2], projection=ccrs.PlateCarree())
        axes.append(ax)

        setup_map_ax(ax, show_left_labels=(i % 2 == 0))

        if p_var not in spatial_data:
            ax.set_title(f"{panel_titles_corr.get(p_var, f'Chl-a vs {p_var}')} (no data)",
                         fontsize=11, fontweight="bold", pad=8)
            continue

        da_phys = subset_season(spatial_data[p_var], season=season)

        common_time = np.intersect1d(da_chl.time.values, da_phys.time.values)
        if len(common_time) < 10:
            ax.set_title(f"{panel_titles_corr.get(p_var, f'Chl-a vs {p_var}')} (insufficient data)",
                         fontsize=11, fontweight="bold", pad=8)
            continue

        chl_sync  = da_chl.sel(time=common_time).interp_like(grid_med).chunk({"time": -1})
        phys_sync = da_phys.sel(time=common_time).interp_like(grid_med).chunk({"time": -1})

        if method == "pearson":
            r_map, p_map, _ = pearson_r_p_map(chl_sync, phys_sync)
        else:
            r_map, p_map, _ = spearman_r_p_map_rank(chl_sync, phys_sync)

        frac = sig_fraction(p_map, alpha=0.05)
        r_g, p_g, _ = area_mean_anomaly_corr(da_chl, da_phys, method=method)

        im = r_map.plot(
            ax=ax, transform=ccrs.PlateCarree(),
            cmap=cmap, add_colorbar=False,
            vmin=vmin, vmax=vmax
        )
        ims.append(im)

        # Contours at ±0.5 to help the reviewer read the map
        try:
            lons = r_map.coords["lon"].values if "lon" in r_map.coords else r_map.coords["longitude"].values
            lats = r_map.coords["lat"].values if "lat" in r_map.coords else r_map.coords["latitude"].values
            ax.contour(lons, lats, r_map.values,
                       levels=[-0.5, 0.5],
                       colors=["#1a1aff", "#cc0000"],
                       linewidths=1.2,
                       transform=ccrs.PlateCarree())
        except Exception:
            pass

        base = panel_titles_corr.get(p_var, f"Chl-a vs {p_var}")
        if np.isfinite(r_g) and np.isfinite(p_g):
            subtitle = f"sig(p<0.05)={frac:.1f}% | global r={r_g:.2f}, p={p_g:.3g}"
        else:
            subtitle = f"sig(p<0.05)={frac:.1f}% | global: NA"

        ax.set_title(f"{base} | {subtitle}", fontsize=10.5, fontweight="bold", pad=8)

    # Color bar in its own axis (column 3), not attached to b/d
    if len(ims) > 0:
        cax = fig.add_subplot(gs[:, 2])
        cbar = fig.colorbar(ims[-1], cax=cax, orientation="vertical")
        cbar.set_label(cbar_label, fontsize=11)
        cbar.ax.tick_params(labelsize=10)

    plt.subplots_adjust(top=0.94, bottom=0.06, left=0.04, right=0.96)
    plt.show()


In [ ]:
# Pearson correlation, full period (2002-2025)
plot_correlation_2x2(method="pearson", season=None)

In [ ]:
# Pearson correlation, dry season (Jun-Nov)
plot_correlation_2x2(method="pearson", season="dry")

In [ ]:
# Pearson correlation, wet season (Dec-May)
plot_correlation_2x2(method="pearson", season="wet")

In [ ]:
# Spearman correlation, full period (2002-2025)
plot_correlation_2x2(method="spearman", season=None)

In [ ]:
# Spearman correlation, dry season (Jun-Nov)
plot_correlation_2x2(method="spearman", season="dry")

In [ ]:
# Spearman correlation, wet season (Dec-May)
plot_correlation_2x2(method="spearman", season="wet")